<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Deep Learning & Neural Networks** · Week 1 · Neural Networks from Scratch · Lesson 4 of 24

# Backpropagation: How Networks Learn

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/deep-learning-neural-networks).

**Learning objective:** Explain backpropagation with the chain rule, and write a two-layer neural network in NumPy that learns the curved two-moons pattern.

## The problem backpropagation solves

A network can have millions of weights. For each one we need its gradient: "if I change this weight a little, how does the loss change?" Nudging each weight one by one would take forever. **Backpropagation** computes **all** the gradients in one backward sweep through the network — about the same cost as one forward pass. It is the engine of all deep learning.

## The chain rule in one picture

If the loss depends on the output, the output depends on the hidden layer, and the hidden layer depends on a weight, then:

> how loss changes with the weight = (how loss changes with the output) × (how output changes with the hidden layer) × (how hidden layer changes with the weight)

Backpropagation starts at the loss and walks **backwards**, multiplying these local slopes layer by layer and reusing them — that is why it is fast.

## The network

- Input: 2 numbers (the moon coordinates).
- Hidden layer: 16 ReLU neurons.
- Output: 1 sigmoid neuron (probability of moon 1).
- Loss: binary cross-entropy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=1000, noise=0.2, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, X_test.shape)
plt.scatter(X[:, 0], X[:, 1], c=y, s=8)
plt.title("Two moons: a pattern a straight line cannot separate")
plt.show()

## Forward and backward, step by step

In [ ]:
rng = np.random.default_rng(0)
H = 16
W1 = rng.normal(0, 1, (2, H))          # input -> hidden
b1 = np.zeros(H)
W2 = rng.normal(0, 1, (H, 1)) * np.sqrt(1 / H)   # hidden -> output
b2 = np.zeros(1)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward(X):
    z1 = X @ W1 + b1
    a1 = np.maximum(0, z1)              # ReLU
    z2 = a1 @ W2 + b2
    p = sigmoid(z2).ravel()
    return z1, a1, p

lr = 0.5
losses = []
for epoch in range(2000):
    z1, a1, p = forward(X_train)
    n = len(y_train)
    losses.append(-np.mean(y_train * np.log(p + 1e-9) + (1 - y_train) * np.log(1 - p + 1e-9)))

    # backward pass (chain rule, from the loss back to the first layer)
    dz2 = (p - y_train)[:, None] / n    # loss -> output score
    dW2 = a1.T @ dz2                    # -> weights of layer 2
    db2 = dz2.sum(axis=0)
    da1 = dz2 @ W2.T                    # -> hidden activations
    dz1 = da1 * (z1 > 0)                # -> through ReLU (slope 1 or 0)
    dW1 = X_train.T @ dz1               # -> weights of layer 1
    db1 = dz1.sum(axis=0)

    # gradient descent step
    W1 -= lr * dW1; b1 -= lr * db1
    W2 -= lr * dW2; b2 -= lr * db2

test_acc = ((forward(X_test)[2] > 0.5) == y_test).mean()
print("final training loss:", round(losses[-1], 3))
print("test accuracy:", test_acc)

Read the backward pass from top to bottom: each line is one link of the chain rule. The ReLU passes the gradient through where the neuron was active (z1 > 0) and blocks it where it was off.

The network reaches about **0.96** test accuracy, up from 0.86 with a single neuron. The hidden layer learned to bend the boundary.

## See the curved boundary

In [ ]:
xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.5, 2, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = forward(grid)[2].reshape(xx.shape)
plt.contourf(xx, yy, zz > 0.5, alpha=0.3)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, s=10)
plt.title("Decision boundary learned by a 16-neuron hidden layer")
plt.show()

plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()

## Why random starting weights?

If every weight started at zero, every hidden neuron would compute the same thing and receive the same gradient — they would stay identical forever. Random initial weights **break the symmetry** so each neuron can learn something different. Scaling them sensibly (here by √(1/H)) keeps the signals a healthy size (Lesson 10).

## You will rarely write this again

From the next lesson, **PyTorch** computes the backward pass for you automatically. But now you know what `loss.backward()` really does: exactly the chain-rule steps above, for any network you build.

## Key terms

- **Backpropagation** — computing every gradient in one backward sweep using the chain rule.
- **Chain rule** — multiply the slopes of each step to get the overall slope.
- **Hidden layer** — lets the network learn curved boundaries.
- **Symmetry breaking** — random initial weights so neurons learn different things.

**Takeaway:** Backpropagation walks backwards from the loss, multiplying local slopes, to give every weight its gradient — and a single hidden layer already turns a straight line into a curve.

## Try it yourself

1. Change the hidden layer to 2, 4 and 64 neurons. How does the test accuracy and the boundary change?
2. Start all weights at zero. What happens to training, and why?
3. Add a second hidden layer (16 → 16). Write its backward lines by following the same pattern.
4. Use the gradient-check trick from Lesson 3 on one entry of W1 to confirm your backward pass.

---
Finished? Go back to [Deep Learning & Neural Networks on WAMUGII](https://wamugii.kento.rw/courses/deep-learning-neural-networks), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*